# Esqueletización de peces

Sube una imagen con el widget, se predicen las máscaras de instancia con `runs/yolov8n-seg_merged/weights/best.pt`
y a cada máscara se le aplica una esqueletización morfológica con `cv2`. En el plot se dibuja el borde de cada individuo
(píxeles de la máscara que no tienen sus 8 vecinos dentro de la máscara) y el esqueleto.
Además se busca un marcador ArUco 4x4 y, si aparece, se dibuja su borde en verde.

In [8]:
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from ultralytics import YOLO

WEIGHTS = Path(r"C:\Users\juanm\Documents\GitHub\fish_cv_pfc\runs\yolov8n-seg_merged\weights\best.pt")
CONF = 0.5
IMGSZ = 640
SKELETON_COLOR = (0, 255, 255)     # RGB, cian
BORDER_COLOR = (255, 0, 255)     # RGB, rojo
ARUCO_COLOR = (0, 255, 0)        # RGB, verde
ARUCO_DICT = cv2.aruco.DICT_4X4_1000   # incluye los IDs de 4X4_50/100/250

model = YOLO(WEIGHTS)

## Función: máscaras + esqueletización

In [9]:
KERNEL_3x3 = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
KERNEL_CROSS = cv2.getStructuringElement(cv2.MORPH_CROSS, (3, 3))


def skeletonize(mask):
    """Esqueleto morfológico (Lantuéjoul) con erosiones/aperturas de cv2. mask: uint8 {0,255}."""
    img = mask.copy()
    skel = np.zeros_like(img)
    while cv2.countNonZero(img) > 0:
        eroded = cv2.erode(img, KERNEL_CROSS)
        opened = cv2.dilate(eroded, KERNEL_CROSS)
        skel = cv2.bitwise_or(skel, cv2.subtract(img, opened))
        img = eroded
    return skel


def border(mask):
    """Píxeles de la máscara que no están rodeados por sus 8 vecinos."""
    return cv2.subtract(mask, cv2.erode(mask, KERNEL_3x3, borderType=cv2.BORDER_CONSTANT, borderValue=0))


def fish_skeletons(img_bgr):
    """Predice máscaras de instancia y devuelve (masks, skeletons, borders, confs).
    masks/skeletons/borders: listas de arrays uint8 {0,255}; confs: confianza del modelo por pez."""
    h, w = img_bgr.shape[:2]
    res = model.predict(img_bgr, imgsz=IMGSZ, conf=CONF, retina_masks=True, verbose=False)[0]
    masks, skeletons, borders = [], [], []
    if res.masks is None:
        return masks, skeletons, borders, []
    confs = res.boxes.conf.cpu().numpy().tolist()
    for m in res.masks.data.cpu().numpy():
        m = (m > 0.5).astype(np.uint8) * 255
        if m.shape != (h, w):
            m = cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST)
        masks.append(m)
        skeletons.append(skeletonize(m))
        borders.append(border(m))
    return masks, skeletons, borders, confs


ARUCO_DETECTOR = cv2.aruco.ArucoDetector(
    cv2.aruco.getPredefinedDictionary(ARUCO_DICT), cv2.aruco.DetectorParameters()
)


def detect_aruco(img_bgr):
    """Busca marcadores ArUco 4x4. Devuelve (corners, ids); corners: lista de arrays (4, 2) float32."""
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    corners, ids, _ = ARUCO_DETECTOR.detectMarkers(gray)
    if ids is None:
        return [], []
    return [c.reshape(4, 2) for c in corners], ids.flatten().tolist()


def plot_skeletons(img_bgr, skeletons, borders, confs=(), aruco_corners=(), aruco_ids=(), name_file_output="defecto.png"):
    out = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    for b in borders:
        out[b > 0] = BORDER_COLOR
    for s in skeletons:
        out[s > 0] = SKELETON_COLOR
    thickness = max(2, round(max(out.shape[:2]) / 400))
    font_scale = thickness / 3
    for b, conf in zip(borders, confs):
        ys, xs = np.nonzero(b)
        if len(ys) == 0:
            continue
        txt = f"{conf:.2f}"
        (tw, th), _ = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, font_scale, thickness)
        x = int(np.clip((xs.min() + xs.max()) / 2 - tw / 2, 0, out.shape[1] - tw))
        y = int(max(ys.min() - 2 * thickness, th))   # justo arriba del pez, sin salirse de la imagen
        cv2.putText(out, txt, (x, y), cv2.FONT_HERSHEY_SIMPLEX, font_scale, BORDER_COLOR, thickness)
    for c, i in zip(aruco_corners, aruco_ids):
        cv2.polylines(out, [c.astype(np.int32)], isClosed=True, color=ARUCO_COLOR, thickness=thickness)
        x, y = c[0].astype(int)
        cv2.putText(out, f"id {i}", (x, y - 2 * thickness), cv2.FONT_HERSHEY_SIMPLEX,
                    font_scale, ARUCO_COLOR, thickness)
    plt.figure(figsize=(12, 8))
    plt.imshow(out)
    aruco_txt = f"ArUco {list(aruco_ids)}" if len(aruco_ids) else "sin ArUco"
    plt.title(f"{len(skeletons)} peces | borde + esqueleto | {aruco_txt}")
    plt.axis("off")

    output_name = name_file_output
    plt.savefig(output_name, bbox_inches='tight', dpi=300)
    print(f"Imagen guardada como: {output_name}")


    plt.show()

## Cargar imagen

In [10]:
uploader = widgets.FileUpload(accept="image/*", multiple=False, description="Imagen")
out = widgets.Output()



def on_upload(change):
    if not change["new"]:
        return
    f = change["new"][0]
    img = cv2.imdecode(np.frombuffer(f["content"], np.uint8), cv2.IMREAD_COLOR)   # BGR
    out.clear_output()
    with out:
        if img is None:
            print(f"No se pudo leer {f['name']}")
            return
        _, skeletons, borders, confs = fish_skeletons(img)
        aruco_corners, aruco_ids = detect_aruco(img)
        plot_skeletons(img, skeletons, borders, confs, aruco_corners, aruco_ids, "result.png")

uploader.observe(on_upload, names="value")
display(uploader, out)

FileUpload(value=(), accept='image/*', description='Imagen')

Output()